# Code 6: Conviction Labeling with ATR-based Exit Strategy (FINAL with Label Shift)

## Overview
Generate **conviction labels** (High/Medium/Low/Ignore) for each (stock, date) by simulating buy-and-hold-until-stopped trades using **4× ATR trailing stop-loss** aligned with Code 8c backtest logic.

## ✅ Key Features
1. **Exit logic aligned with Code 8c:**
   - Track peak using `High` (intraday peak)
   - Check exit when intraday `Low` ≤ trigger price
   - Exit at **trigger price** (not Close) — realistic stop-loss order
2. **Uses existing `ATR_14`** from `data_features.parquet` (no duplicate calculation)
3. **Partial ATR** for first ~13 days per stock (where `ATR_14` is NaN)
4. **Transaction cost 0.25%** applied to entry/exit for return calculation only
5. **Stop-loss %** uses RAW entry price (not adjusted)
6. **Force exit at last Close** if stop never hits within dataset
7. **Label shift to prevent look-ahead bias** (NEW):
   - `conviction_label` shifted UP by 1 within each ticker
   - Row at entry_date=t now holds the label for trade at entry_date=t+1
   - Last row of each ticker → label = NaN
   - Result: features at end of day t → predict outcome of trade entered at t+1 close ✓
8. **Input from Google Drive** (parquet for speed)
9. **Outputs auto-download** to your computer's Downloads folder

## Inputs
- `data_features.parquet` (from Code 3) — Google Drive

## Outputs
- `data_exit_all.csv` — Main output (all stocks with shifted conviction labels)
- `data_exit_bpcl_test.csv` — BPCL test results
- `failed_stocks_code6.csv` — Any stocks that failed (if any)

**Estimated runtime:** 30-90 minutes for all stocks

---
## Cell 1: Mount Google Drive & Import Libraries

In [1]:
# Cell 1: Mount Google Drive and import libraries

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from google.colab import files

print("✓ Libraries imported successfully!")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Numpy version: {np.__version__}")
print(f"✓ Drive mounted at /content/drive")

Mounted at /content/drive
✓ Libraries imported successfully!
✓ Pandas version: 2.2.2
✓ Numpy version: 2.0.2
✓ Drive mounted at /content/drive


---
## Cell 2: Define Parameters
**Easily modifiable parameters for experimentation**

In [2]:
# Cell 2: Configuration & Parameters

# ============================================================
# FILE PATHS
# ============================================================
DRIVE_FOLDER = '/content/drive/MyDrive/masters'   # 👈 UPDATE if your folder name is different
INPUT_FILE = f'{DRIVE_FOLDER}/data_features.parquet'

# ============================================================
# CONVICTION LABEL THRESHOLDS (annualized return %)
# ============================================================
CONVICTION_THRESHOLDS = {
    'high_min': 40.0,      # >40% annualized → High
    'medium_min': 20.0,    # 20-40% → Medium
    'low_min': 10.0,       # 10-20% → Low
    # <10% → Ignore
}

# ============================================================
# ATR & TRADING PARAMETERS
# ============================================================
ATR_PERIOD = 14            # Standard 14-day ATR (matches Code 3)
ATR_MULTIPLIER = 4.0       # Stop loss = 4 × ATR
TRANSACTION_COST = 0.25    # 0.25% on both buy and sell

# ============================================================
# LABEL SHIFT (NEW - prevents look-ahead bias)
# ============================================================
# When True, conviction_label is shifted UP by 1 within each ticker.
# This aligns label with features known one day before the trade.
SHIFT_LABEL_FOR_NO_LOOKAHEAD = True

# ============================================================
# DISPLAY CONFIGURATION
# ============================================================
print("=" * 70)
print("CODE 6: CONVICTION LABELING CONFIGURATION")
print("=" * 70)
print(f"Input file:           {INPUT_FILE}")
print("\nConviction Thresholds (Annualized Return):")
print(f"  High:    > {CONVICTION_THRESHOLDS['high_min']}%")
print(f"  Medium:  {CONVICTION_THRESHOLDS['medium_min']}% - {CONVICTION_THRESHOLDS['high_min']}%")
print(f"  Low:     {CONVICTION_THRESHOLDS['low_min']}% - {CONVICTION_THRESHOLDS['medium_min']}%")
print(f"  Ignore:  < {CONVICTION_THRESHOLDS['low_min']}%")
print("\nATR & Trading Parameters:")
print(f"  ATR Period:          {ATR_PERIOD} days")
print(f"  ATR Multiplier:      {ATR_MULTIPLIER}× (stop-loss = {ATR_MULTIPLIER} × ATR)")
print(f"  Transaction Cost:    {TRANSACTION_COST}% (on buy and sell)")
print(f"  Shift Label by -1:   {SHIFT_LABEL_FOR_NO_LOOKAHEAD}  (prevents look-ahead bias)")
print("\nExit Logic (aligned with Code 8c):")
print(f"  • Stop-loss % = ({ATR_MULTIPLIER} × ATR) / entry_price_raw × 100")
print(f"  • Peak tracked using intraday High")
print(f"  • Trigger = Peak × (1 - stop_loss_pct/100)")
print(f"  • Exit when intraday Low ≤ Trigger")
print(f"  • Exit price = Trigger price (realistic stop order)")
print(f"  • If never hit: Force exit at last Close in dataset")
print("=" * 70)

CODE 6: CONVICTION LABELING CONFIGURATION
Input file:           /content/drive/MyDrive/masters/data_features.parquet

Conviction Thresholds (Annualized Return):
  High:    > 40.0%
  Medium:  20.0% - 40.0%
  Low:     10.0% - 20.0%
  Ignore:  < 10.0%

ATR & Trading Parameters:
  ATR Period:          14 days
  ATR Multiplier:      4.0× (stop-loss = 4.0 × ATR)
  Transaction Cost:    0.25% (on buy and sell)
  Shift Label by -1:   True  (prevents look-ahead bias)

Exit Logic (aligned with Code 8c):
  • Stop-loss % = (4.0 × ATR) / entry_price_raw × 100
  • Peak tracked using intraday High
  • Trigger = Peak × (1 - stop_loss_pct/100)
  • Exit when intraday Low ≤ Trigger
  • Exit price = Trigger price (realistic stop order)
  • If never hit: Force exit at last Close in dataset


---
## Cell 3: Load Data from Google Drive
Reads **only required columns** from parquet for speed & memory efficiency

In [3]:
# Cell 3: Load data_features.parquet (only required columns)

REQUIRED_COLS = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'ATR_14']

print(f"Loading file: {INPUT_FILE}")
print(f"Reading only required columns: {REQUIRED_COLS}")
print("\nThis is much faster than loading all 172 columns...")

try:
    df_input = pd.read_parquet(INPUT_FILE, columns=REQUIRED_COLS)
    df_input['Date'] = pd.to_datetime(df_input['Date'])
    df_input = df_input.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    print(f"\n✓ Data loaded successfully!")
    print(f"  Shape: {df_input.shape}")
    print(f"  Memory usage: {df_input.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"  Columns: {list(df_input.columns)}")
    print(f"  Date range: {df_input['Date'].min()} to {df_input['Date'].max()}")
    print(f"  Unique tickers: {df_input['Ticker'].nunique()}")

    GLOBAL_LAST_DATE = df_input['Date'].max()
    print(f"\n*** GLOBAL LAST DATE (for force exit): {GLOBAL_LAST_DATE} ***")

    print(f"\nATR_14 column stats:")
    print(f"  Total rows: {len(df_input)}")
    print(f"  NaN count: {df_input['ATR_14'].isna().sum()} ({df_input['ATR_14'].isna().sum()/len(df_input)*100:.2f}%)")
    print(f"  Mean ATR_14: {df_input['ATR_14'].mean():.2f}")
    print(f"  (NaN values are normal — first ~13 days per stock; we'll fill with partial ATR)")

    print(f"\nFirst few rows:")
    print(df_input.head())

except FileNotFoundError:
    print(f"\n❌ ERROR: File not found at {INPUT_FILE}")
    print("\nPlease check:")
    print("1. Is Google Drive mounted? (Cell 1)")
    print("2. Is the file at the path above?")
    print("3. Update DRIVE_FOLDER in Cell 2 if needed")
except Exception as e:
    print(f"\n❌ ERROR loading file: {str(e)}")

Loading file: /content/drive/MyDrive/masters/data_features.parquet
Reading only required columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'ATR_14']

This is much faster than loading all 172 columns...

✓ Data loaded successfully!
  Shape: (2417660, 8)
  Memory usage: 203.24 MB
  Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'ATR_14']
  Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
  Unique tickers: 570

*** GLOBAL LAST DATE (for force exit): 2026-06-12 00:00:00 ***

ATR_14 column stats:
  Total rows: 2417660
  NaN count: 0 (0.00%)
  Mean ATR_14: 27.41
  (NaN values are normal — first ~13 days per stock; we'll fill with partial ATR)

First few rows:
        Date     Ticker         Open         High         Low       Close  \
0 2007-01-02  3IINFOLTD   962.630615   977.636169  858.319092  859.410400   
1 2007-01-03  3IINFOLTD   983.365723  1002.645569  960.630005  976.272156   
2 2007-01-04  3IINFOLTD   964.904175   993.096497  954.900

---
## Cell 4: Fill ATR NaN with Partial ATR
For first ~13 days where `ATR_14` is NaN, compute partial ATR using available days.

In [4]:
# Cell 4: Function to fill NaN ATR values with partial calculation

def fill_partial_atr(stock_df, atr_period=14):
    """
    Fill NaN values in ATR_14 column using partial ATR calculation.

    For rows where ATR_14 is NaN (typically first ~13 days of stock data),
    compute ATR using whatever data is available up to that point.
    """
    df = stock_df.copy().sort_values('Date').reset_index(drop=True)

    df['ATR_Source'] = np.where(df['ATR_14'].isna(), 'Partial_Calc', 'TA_Library')

    nan_mask = df['ATR_14'].isna()

    if not nan_mask.any():
        return df

    # Compute True Range for all rows
    df['_prev_close'] = df['Close'].shift(1)
    df['_tr1'] = df['High'] - df['Low']
    df['_tr2'] = (df['High'] - df['_prev_close']).abs()
    df['_tr3'] = (df['Low'] - df['_prev_close']).abs()
    df['_TR'] = df[['_tr1', '_tr2', '_tr3']].max(axis=1)

    df.loc[0, '_TR'] = df.loc[0, '_tr1']

    # Partial ATR for NaN rows
    for i in df.index[nan_mask]:
        lookback = min(i + 1, atr_period)
        start_idx = max(0, i - lookback + 1)
        df.loc[i, 'ATR_14'] = df.loc[start_idx:i, '_TR'].mean()

    df = df.drop(['_prev_close', '_tr1', '_tr2', '_tr3', '_TR'], axis=1)

    return df

print("✓ Partial ATR function defined")
print("  • Where ATR_14 exists → keep (marked 'TA_Library')")
print("  • Where ATR_14 is NaN → fill with partial calc (marked 'Partial_Calc')")

✓ Partial ATR function defined
  • Where ATR_14 exists → keep (marked 'TA_Library')
  • Where ATR_14 is NaN → fill with partial calc (marked 'Partial_Calc')


---
## Cell 5: Buy-Hold-Exit Simulation Function (ALIGNED WITH CODE 8c)
1. Stop-loss % from **raw entry price**
2. Peak tracked using **intraday High**
3. Exit when **intraday Low ≤ Trigger**
4. Exit price = **Trigger price**
5. Returns calculated using transaction-cost-adjusted prices

In [5]:
# Cell 5: Buy-Hold-Exit Simulation (aligned with Code 8c logic)

def simulate_exits_for_stock(stock_df, ticker,
                              atr_multiplier=4.0,
                              transaction_cost_pct=0.25,
                              dataset_last_date=None):
    """
    Simulate buy on each date and track exit using Code 8c logic.
    """
    df = stock_df.copy().sort_values('Date').reset_index(drop=True)

    if dataset_last_date is None:
        dataset_last_date = df['Date'].max()

    tc_decimal = transaction_cost_pct / 100.0

    # Numpy arrays for speed
    dates = df['Date'].values
    highs = df['High'].values
    lows = df['Low'].values
    closes = df['Close'].values
    atrs = df['ATR_14'].values
    atr_sources = df['ATR_Source'].values
    n = len(df)

    results = []

    for entry_idx in range(n):
        entry_date = dates[entry_idx]
        entry_price_raw = closes[entry_idx]
        atr_at_entry = atrs[entry_idx]
        atr_source = atr_sources[entry_idx]

        if pd.isna(atr_at_entry):
            continue
        if pd.isna(entry_price_raw) or entry_price_raw <= 0:
            continue

        # Stop-loss using RAW price
        stop_loss_threshold_rupees = atr_multiplier * atr_at_entry
        stop_loss_pct = (stop_loss_threshold_rupees / entry_price_raw) * 100

        # Forward simulation (Code 8c logic)
        running_max_price = entry_price_raw
        exit_found = False
        exit_idx = None
        exit_price_raw = None
        exit_reason = None

        for future_idx in range(entry_idx + 1, n):
            day_high = highs[future_idx]
            day_low = lows[future_idx]

            # Peak uses INTRADAY HIGH
            if day_high > running_max_price:
                running_max_price = day_high

            trigger_price = running_max_price * (1 - stop_loss_pct / 100)

            # Exit when intraday LOW touches trigger
            if day_low <= trigger_price:
                exit_idx = future_idx
                exit_price_raw = trigger_price  # Exit AT trigger
                exit_found = True
                exit_reason = 'stop_loss_hit'
                break

        # Force exit at last Close if never hit
        if not exit_found:
            if entry_idx + 1 < n:
                exit_idx = n - 1
                exit_price_raw = closes[exit_idx]
                exit_reason = 'forced_period_end'
            else:
                exit_idx = entry_idx
                exit_price_raw = entry_price_raw
                exit_reason = 'no_future_data'

        exit_date = dates[exit_idx]

        # Apply transaction costs (for return calc only)
        entry_price_adjusted = entry_price_raw * (1 + tc_decimal)
        exit_price_adjusted = exit_price_raw * (1 - tc_decimal)

        days_held = (pd.Timestamp(exit_date) - pd.Timestamp(entry_date)).days
        if days_held == 0:
            days_held = 1

        raw_return_pct = ((exit_price_adjusted / entry_price_adjusted) - 1) * 100
        annualized_return_pct = (((exit_price_adjusted / entry_price_adjusted) ** (365.0 / days_held)) - 1) * 100

        max_price_achieved = running_max_price
        drawdown_at_exit_rupees = max_price_achieved - exit_price_raw
        drawdown_at_exit_pct = (drawdown_at_exit_rupees / max_price_achieved) * 100 if max_price_achieved > 0 else 0

        if exit_reason == 'no_future_data':
            data_quality_flag = 'no_future_data'
        elif exit_reason == 'forced_period_end' and exit_date < dataset_last_date:
            data_quality_flag = 'stock_ended_before_dataset_end'
        else:
            data_quality_flag = 'OK'

        results.append({
            'Ticker': ticker,
            'entry_date': entry_date,
            'entry_price_raw': entry_price_raw,
            'entry_price_adjusted': entry_price_adjusted,
            'atr_at_entry': atr_at_entry,
            'atr_source': atr_source,
            'stop_loss_threshold_rupees': stop_loss_threshold_rupees,
            'stop_loss_pct': stop_loss_pct,
            'exit_date': exit_date,
            'exit_price_raw': exit_price_raw,
            'exit_price_adjusted': exit_price_adjusted,
            'exit_reason': exit_reason,
            'max_price_achieved': max_price_achieved,
            'drawdown_at_exit_rupees': drawdown_at_exit_rupees,
            'drawdown_at_exit_pct': drawdown_at_exit_pct,
            'days_held': days_held,
            'transaction_cost_pct': transaction_cost_pct,
            'raw_return_pct': raw_return_pct,
            'annualized_return_pct': annualized_return_pct,
            'data_quality_flag': data_quality_flag
        })

    return pd.DataFrame(results)

print("✓ Simulation function defined (aligned with Code 8c)")
print("\nExit Logic:")
print("  • stop_loss_pct = (4 × ATR) / entry_price_RAW × 100")
print("  • Peak tracked using intraday High")
print("  • Exit when intraday Low ≤ Trigger")
print("  • Exit price = Trigger price")
print("  • Force exit at last Close if never hit")

✓ Simulation function defined (aligned with Code 8c)

Exit Logic:
  • stop_loss_pct = (4 × ATR) / entry_price_RAW × 100
  • Peak tracked using intraday High
  • Exit when intraday Low ≤ Trigger
  • Exit price = Trigger price
  • Force exit at last Close if never hit


---
## Cell 6: Assign Conviction Labels

In [6]:
# Cell 6: Assign conviction labels based on annualized return

def assign_conviction_labels(results_df, thresholds):
    """
    Assign conviction label based on annualized return.
    """
    df = results_df.copy()

    conditions = [
        df['annualized_return_pct'] >= thresholds['high_min'],
        df['annualized_return_pct'] >= thresholds['medium_min'],
        df['annualized_return_pct'] >= thresholds['low_min'],
    ]
    choices = ['High', 'Medium', 'Low']
    df['conviction_label'] = np.select(conditions, choices, default='Ignore')

    return df

print("✓ Conviction labeling function defined")

✓ Conviction labeling function defined


---
## Cell 7: Shift Conviction Label (NEW — Prevents Look-Ahead Bias) ⭐

**The Problem:** Currently, label at row `entry_date=t` describes the trade entered at t. If Code 7 merges on `entry_date=t`, the model would use features known at end of day t — but we'd need to make the buy decision BEFORE close of day t.

**The Fix:** Shift `conviction_label` UP by 1 within each ticker so the label at row `entry_date=t-1` describes the trade at t (next trading day).

**Result:**
- Code 7 merges `features.Date=t-1` with `entry_date=t-1` → label retrieved is for trade at t ✓
- Features known at end of t-1 → predict outcome of trade entered at t (close) ✓ No look-ahead bias
- Last row per ticker → `conviction_label = NaN` (no future trade to predict)

In [7]:
# Cell 7: Shift conviction_label for look-ahead bias prevention

def shift_conviction_labels(results_df, do_shift=True):
    """
    Shift conviction_label UP by 1 within each ticker group.

    - Row at entry_date=t-1 now holds label for trade at entry_date=t
    - Last row per ticker → NaN
    """
    df = results_df.copy()

    if not do_shift:
        return df

    df = df.sort_values(['Ticker', 'entry_date']).reset_index(drop=True)

    # Shift UP by 1 within each ticker
    # shift(-1) → new[i] = old[i+1]
    df['conviction_label'] = df.groupby('Ticker')['conviction_label'].shift(-1)

    return df

print("✓ Label shift function defined")
print("\nWhat this does:")
print("  • Shifts 'conviction_label' UP by 1 within each ticker")
print("  • New conviction_label = label of NEXT trading day's trade for this stock")
print("  • Last row per ticker → conviction_label = NaN")
print("  • Code 7 merges on features.Date == data_exit_all.entry_date")
print("  • Result: features at t-1 → predict outcome of trade at t (no look-ahead)")

✓ Label shift function defined

What this does:
  • Shifts 'conviction_label' UP by 1 within each ticker
  • New conviction_label = label of NEXT trading day's trade for this stock
  • Last row per ticker → conviction_label = NaN
  • Code 7 merges on features.Date == data_exit_all.entry_date
  • Result: features at t-1 → predict outcome of trade at t (no look-ahead)


---
## Cell 8: Test with BPCL (Single Stock)
Always test with one stock first to verify logic and label shift

In [8]:
# Cell 8: Test with BPCL

BPCL_TICKER = 'BPCL'  # 👈 Update if your data uses different ticker format

df_bpcl_raw = df_input[df_input['Ticker'] == BPCL_TICKER].copy()

print("=" * 70)
print(f"TESTING WITH: {BPCL_TICKER}")
print("=" * 70)

if len(df_bpcl_raw) == 0:
    print(f"\n⚠️ No data found for '{BPCL_TICKER}'")
    print("\nAvailable tickers containing 'BPCL':")
    similar = df_input[df_input['Ticker'].str.contains('BPCL', case=False, na=False)]['Ticker'].unique()
    print(similar)
else:
    print(f"Total rows: {len(df_bpcl_raw)}")
    print(f"Date range: {df_bpcl_raw['Date'].min()} to {df_bpcl_raw['Date'].max()}")

    # Step 1: Fill partial ATR
    print("\nStep 1: Filling NaN ATR values...")
    df_bpcl = fill_partial_atr(df_bpcl_raw, atr_period=ATR_PERIOD)
    print(f"✓ ATR Source: {df_bpcl['ATR_Source'].value_counts().to_dict()}")

    # Step 2: Simulate exits
    print("\nStep 2: Simulating buy-hold-exit...")
    results_bpcl = simulate_exits_for_stock(
        stock_df=df_bpcl,
        ticker=BPCL_TICKER,
        atr_multiplier=ATR_MULTIPLIER,
        transaction_cost_pct=TRANSACTION_COST,
        dataset_last_date=GLOBAL_LAST_DATE
    )
    print(f"✓ Simulated {len(results_bpcl)} trades")

    # Step 3: Assign labels
    print("\nStep 3: Assigning conviction labels...")
    results_bpcl = assign_conviction_labels(results_bpcl, CONVICTION_THRESHOLDS)
    print(f"✓ Labels assigned")

    # Step 4: Shift labels (prevents look-ahead bias)
    print("\nStep 4: Shifting conviction_label by -1...")
    results_bpcl = shift_conviction_labels(results_bpcl, do_shift=SHIFT_LABEL_FOR_NO_LOOKAHEAD)
    print(f"✓ Labels shifted")

    # Verify shift
    last_label = results_bpcl.iloc[-1]['conviction_label']
    nan_count = results_bpcl['conviction_label'].isna().sum()

    print(f"\nVerification:")
    print(f"  Last row conviction_label:     {last_label}  (should be NaN)")
    print(f"  NaN count in conviction_label: {nan_count}  (should be 1)")

    print(f"\nFirst 5 trades:")
    cols = ['entry_date', 'annualized_return_pct', 'conviction_label']
    print(results_bpcl[cols].head(5).to_string(index=False))

    print(f"\nLast 5 trades:")
    print(results_bpcl[cols].tail(5).to_string(index=False))

TESTING WITH: BPCL
Total rows: 4797
Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00

Step 1: Filling NaN ATR values...
✓ ATR Source: {'TA_Library': 4797}

Step 2: Simulating buy-hold-exit...
✓ Simulated 4797 trades

Step 3: Assigning conviction labels...
✓ Labels assigned

Step 4: Shifting conviction_label by -1...
✓ Labels shifted

Verification:
  Last row conviction_label:     nan  (should be NaN)
  NaN count in conviction_label: 1  (should be 1)

First 5 trades:
entry_date  annualized_return_pct conviction_label
2007-01-02          -5.260518e+01             High
2007-01-03           1.529451e+09             High
2007-01-04           2.594170e+07             High
2007-01-05           3.904105e+02             High
2007-01-08           7.020747e+02           Ignore

Last 5 trades:
entry_date  annualized_return_pct conviction_label
2026-06-08               7.739711             High
2026-06-09             235.648819             High
2026-06-10             425.093506             Hi

---
## Cell 9: Analyze BPCL Results

In [9]:
# Cell 9: BPCL Detailed Analysis

print("=" * 80)
print(f"BPCL ({BPCL_TICKER}) — DETAILED ANALYSIS")
print("=" * 80)

# Overall stats
print("\n1. OVERALL STATISTICS")
print("-" * 80)
print(f"Total simulated trades: {len(results_bpcl)}")
print(f"\nReturn Stats (AFTER 0.25% transaction cost):")
print(f"  Mean Annualized:   {results_bpcl['annualized_return_pct'].mean():>8.2f}%")
print(f"  Median Annualized: {results_bpcl['annualized_return_pct'].median():>8.2f}%")
print(f"\nHolding Period:")
print(f"  Mean Days Held:   {results_bpcl['days_held'].mean():>8.1f}")
print(f"  Median Days Held: {results_bpcl['days_held'].median():>8.1f}")
print(f"\nStop-Loss %:")
print(f"  Mean:   {results_bpcl['stop_loss_pct'].mean():>6.2f}%")
print(f"  Median: {results_bpcl['stop_loss_pct'].median():>6.2f}%")

# SHIFTED label distribution (used by model)
print("\n2. CONVICTION LABEL DISTRIBUTION (SHIFTED — used by ML model)")
print("-" * 80)
label_counts = results_bpcl['conviction_label'].value_counts(dropna=False)
n_total = len(results_bpcl)
for label in ['High', 'Medium', 'Low', 'Ignore']:
    count = label_counts.get(label, 0)
    pct = count / n_total * 100
    print(f"  {label:8s}: {count:>5d} ({pct:>5.2f}%)")
nan_count = results_bpcl['conviction_label'].isna().sum()
print(f"  NaN     : {nan_count:>5d} ({nan_count/n_total*100:>5.2f}%)  ← last row of ticker")

# Exit reasons
print("\n3. EXIT REASON DISTRIBUTION")
print("-" * 80)
for reason, count in results_bpcl['exit_reason'].value_counts().items():
    pct = (count / len(results_bpcl)) * 100
    print(f"  {reason:25s}: {count:>5d} ({pct:>5.2f}%)")

# ATR source
print("\n4. ATR SOURCE BREAKDOWN")
print("-" * 80)
for src, count in results_bpcl['atr_source'].value_counts().items():
    pct = (count / len(results_bpcl)) * 100
    print(f"  {src:25s}: {count:>5d} ({pct:>5.2f}%)")

# Sample High trades
print("\n5. SAMPLE HIGH CONVICTION TRADES (using SHIFTED label, first 5)")
print("-" * 80)
high_trades = results_bpcl[results_bpcl['conviction_label'] == 'High'].head(5)
if len(high_trades) > 0:
    cols = ['entry_date', 'exit_date', 'days_held', 'stop_loss_pct',
            'annualized_return_pct', 'conviction_label']
    print("Note: conviction_label here = label for trade entered at NEXT entry_date")
    print()
    print(high_trades[cols].to_string(index=False))
else:
    print("  No High conviction trades found")

BPCL (BPCL) — DETAILED ANALYSIS

1. OVERALL STATISTICS
--------------------------------------------------------------------------------
Total simulated trades: 4797

Return Stats (AFTER 0.25% transaction cost):
  Mean Annualized:   855249024.00%
  Median Annualized:   -10.10%

Holding Period:
  Mean Days Held:      154.2
  Median Days Held:     76.0

Stop-Loss %:
  Mean:    16.15%
  Median:  14.72%

2. CONVICTION LABEL DISTRIBUTION (SHIFTED — used by ML model)
--------------------------------------------------------------------------------
  High    :   798 (16.64%)
  Medium  :   676 (14.09%)
  Low     :   384 ( 8.01%)
  Ignore  :  2938 (61.25%)
  NaN     :     1 ( 0.02%)  ← last row of ticker

3. EXIT REASON DISTRIBUTION
--------------------------------------------------------------------------------
  stop_loss_hit            :  4715 (98.29%)
  forced_period_end        :    81 ( 1.69%)
  no_future_data           :     1 ( 0.02%)

4. ATR SOURCE BREAKDOWN
------------------------------

---
## Cell 10: Multi-Stock Processing Function

In [10]:
# Cell 10: Multi-stock processing function

def process_multiple_stocks(df_input, tickers_list,
                             atr_period=14,
                             atr_multiplier=4.0,
                             transaction_cost_pct=0.25,
                             dataset_last_date=None,
                             thresholds=None,
                             do_shift_label=True):
    """
    Process all stocks: fill ATR, simulate exits, assign labels, shift labels.
    """
    if thresholds is None:
        thresholds = CONVICTION_THRESHOLDS
    if dataset_last_date is None:
        dataset_last_date = df_input['Date'].max()

    all_results = []
    failed_stocks = []

    print(f"Processing {len(tickers_list)} stocks...")
    print(f"Dataset last date: {dataset_last_date}")
    print(f"Label shift enabled: {do_shift_label}")
    print("=" * 70)

    grouped = df_input.groupby('Ticker')

    for ticker in tqdm(tickers_list, desc="Processing"):
        try:
            if ticker not in grouped.groups:
                failed_stocks.append({'ticker': ticker, 'reason': 'no_data'})
                continue

            stock_df = grouped.get_group(ticker).copy()

            if len(stock_df) == 0:
                failed_stocks.append({'ticker': ticker, 'reason': 'no_data'})
                continue

            # Step 1: Fill partial ATR
            stock_df = fill_partial_atr(stock_df, atr_period=atr_period)

            # Step 2: Simulate exits
            results = simulate_exits_for_stock(
                stock_df=stock_df,
                ticker=ticker,
                atr_multiplier=atr_multiplier,
                transaction_cost_pct=transaction_cost_pct,
                dataset_last_date=dataset_last_date
            )

            if len(results) == 0:
                failed_stocks.append({'ticker': ticker, 'reason': 'no_simulated_trades'})
                continue

            # Step 3: Assign labels
            results = assign_conviction_labels(results, thresholds)

            # Step 4: Shift labels (per-ticker, no leakage across tickers)
            results = shift_conviction_labels(results, do_shift=do_shift_label)

            all_results.append(results)

        except Exception as e:
            failed_stocks.append({'ticker': ticker, 'reason': str(e)[:200]})
            continue

    if len(all_results) > 0:
        final_results = pd.concat(all_results, ignore_index=True)
    else:
        final_results = pd.DataFrame()

    print(f"\n✓ Successfully processed: {len(all_results)} stocks")
    print(f"⚠ Failed: {len(failed_stocks)} stocks")
    print(f"✓ Total records: {len(final_results)}")

    return final_results, failed_stocks

print("✓ Multi-stock processing function defined")
print("  (includes label shift step within each ticker processing)")

✓ Multi-stock processing function defined
  (includes label shift step within each ticker processing)


---
## Cell 11: Process All Stocks
**⏱️ Takes 30-90 minutes for ~2000 stocks**

In [11]:
# Cell 11: Process ALL stocks

print("=" * 80)
print("PROCESSING ALL STOCKS")
print("=" * 80)

all_tickers = df_input['Ticker'].unique()
print(f"Total unique stocks: {len(all_tickers)}")
print(f"Estimated runtime: 30-90 minutes\n")

import time
start_time = time.time()

results_all_stocks, failed_list = process_multiple_stocks(
    df_input=df_input,
    tickers_list=all_tickers,
    atr_period=ATR_PERIOD,
    atr_multiplier=ATR_MULTIPLIER,
    transaction_cost_pct=TRANSACTION_COST,
    dataset_last_date=GLOBAL_LAST_DATE,
    thresholds=CONVICTION_THRESHOLDS,
    do_shift_label=SHIFT_LABEL_FOR_NO_LOOKAHEAD
)

elapsed = (time.time() - start_time) / 60
print(f"\n⏱ Total time: {elapsed:.1f} minutes")

# Summary
print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print(f"Total records: {len(results_all_stocks):,}")
print(f"Unique stocks: {results_all_stocks['Ticker'].nunique()}")
if len(results_all_stocks) > 0:
    print(f"Date range: {results_all_stocks['entry_date'].min()} to {results_all_stocks['entry_date'].max()}")

    # NaN check
    nan_labels = results_all_stocks['conviction_label'].isna().sum()
    n_tickers = results_all_stocks['Ticker'].nunique()
    print(f"\nNaN labels (expect ~{n_tickers} = 1 per ticker): {nan_labels:,}")
    if nan_labels == n_tickers:
        print("  ✓ Label shift correct!")

    print("\nConviction Label Distribution (SHIFTED — model target):")
    conv_dist = results_all_stocks['conviction_label'].value_counts(dropna=False)
    n_total = len(results_all_stocks)
    for label in ['High', 'Medium', 'Low', 'Ignore']:
        count = conv_dist.get(label, 0)
        pct = count / n_total * 100
        print(f"  {label:8s}: {count:>8,d} ({pct:>5.2f}%)")
    print(f"  NaN     : {nan_labels:>8,d} ({nan_labels/n_total*100:>5.2f}%)")

    print("\nExit Reason Distribution:")
    for reason, count in results_all_stocks['exit_reason'].value_counts().items():
        pct = (count / len(results_all_stocks)) * 100
        print(f"  {reason:25s}: {count:>8,d} ({pct:>5.2f}%)")

    print("\nReturn Stats by Conviction (shifted — model target):")
    for label in ['High', 'Medium', 'Low', 'Ignore']:
        subset = results_all_stocks[results_all_stocks['conviction_label'] == label]
        if len(subset) > 0:
            print(f"\n  {label}:")
            print(f"    Count:              {len(subset):>8,d}")
            print(f"    Mean Annualized:  {subset['annualized_return_pct'].mean():>9.2f}%")
            print(f"    Median Annualized:{subset['annualized_return_pct'].median():>9.2f}%")
            print(f"    Median Days Held: {subset['days_held'].median():>9.1f}")

PROCESSING ALL STOCKS
Total unique stocks: 570
Estimated runtime: 30-90 minutes

Processing 570 stocks...
Dataset last date: 2026-06-12 00:00:00
Label shift enabled: True


Processing: 100%|██████████| 570/570 [04:13<00:00,  2.25it/s]



✓ Successfully processed: 570 stocks
⚠ Failed: 0 stocks
✓ Total records: 2417660

⏱ Total time: 4.2 minutes

OVERALL SUMMARY
Total records: 2,417,660
Unique stocks: 570
Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00

NaN labels (expect ~570 = 1 per ticker): 570
  ✓ Label shift correct!

Conviction Label Distribution (SHIFTED — model target):
  High    :  603,804 (24.97%)
  Medium  :  183,820 ( 7.60%)
  Low     :  107,877 ( 4.46%)
  Ignore  : 1,521,589 (62.94%)
  NaN     :      570 ( 0.02%)

Exit Reason Distribution:
  stop_loss_hit            : 2,374,068 (98.20%)
  forced_period_end        :   43,022 ( 1.78%)
  no_future_data           :      570 ( 0.02%)

Return Stats by Conviction (shifted — model target):

  High:
    Count:               603,804
    Mean Annualized:        inf%
    Median Annualized:   101.33%
    Median Days Held:     123.0

  Medium:
    Count:               183,820
    Mean Annualized:        inf%
    Median Annualized:    28.14%
    Median Days Held:  

---
## Cell 12: Save & Auto-Download Output Files

In [12]:
# Cell 12: Save output files locally + auto-download

import os

print("=" * 80)
print("SAVING OUTPUT FILES (LOCAL TO COLAB)")
print("=" * 80)

files_to_download = []

# 1. Main output
main_filename = 'data_exit_all.csv'
results_all_stocks.to_csv(main_filename, index=False)
size_mb = os.path.getsize(main_filename) / (1024 * 1024)
print(f"✓ {main_filename} saved ({len(results_all_stocks):,} records, {size_mb:.2f} MB)")
files_to_download.append(main_filename)

# 2. BPCL test
bpcl_filename = 'data_exit_bpcl_test.csv'
results_bpcl.to_csv(bpcl_filename, index=False)
size_kb = os.path.getsize(bpcl_filename) / 1024
print(f"✓ {bpcl_filename} saved ({len(results_bpcl):,} records, {size_kb:.1f} KB)")
files_to_download.append(bpcl_filename)

# 3. Failed stocks (if any)
if len(failed_list) > 0:
    failed_filename = 'failed_stocks_code6.csv'
    pd.DataFrame(failed_list).to_csv(failed_filename, index=False)
    print(f"✓ {failed_filename} saved ({len(failed_list)} stocks)")
    files_to_download.append(failed_filename)
else:
    print("✓ No failed stocks — perfect run!")

# Auto-download
print("\n" + "=" * 80)
print("AUTO-DOWNLOADING TO YOUR DOWNLOADS FOLDER")
print("=" * 80)

for f in files_to_download:
    print(f"⬇ Downloading {f}...")
    files.download(f)

print("\n✅ ALL FILES DOWNLOADED!")
print("\nFiles in your browser's Downloads folder:")
for f in files_to_download:
    print(f"  • {f}")

SAVING OUTPUT FILES (LOCAL TO COLAB)
✓ data_exit_all.csv saved (2,417,660 records, 435.93 MB)
✓ data_exit_bpcl_test.csv saved (4,797 records, 878.7 KB)
✓ No failed stocks — perfect run!

AUTO-DOWNLOADING TO YOUR DOWNLOADS FOLDER
⬇ Downloading data_exit_all.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇ Downloading data_exit_bpcl_test.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ ALL FILES DOWNLOADED!

Files in your browser's Downloads folder:
  • data_exit_all.csv
  • data_exit_bpcl_test.csv


---
## Cell 13: Verify Output File

In [13]:
# Cell 13: Verify saved output

print("=" * 80)
print("VERIFYING data_exit_all.csv")
print("=" * 80)

df_verify = pd.read_csv('data_exit_all.csv', parse_dates=['entry_date', 'exit_date'])

print(f"\nShape: {df_verify.shape}")
print(f"\nColumns ({len(df_verify.columns)}):")
for col in df_verify.columns:
    print(f"  • {col}: {df_verify[col].dtype}")

print(f"\nFirst 5 rows:")
print(df_verify.head())

print(f"\nLast 5 rows:")
print(df_verify.tail())

print(f"\nMissing values:")
missing = df_verify.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
    print("\nNote: NaN in 'conviction_label' is EXPECTED for last row of each ticker (label shift)")
else:
    print("  ✓ No missing values")

print(f"\nKey Statistics:")
print(f"  Unique tickers:           {df_verify['Ticker'].nunique()}")
print(f"  Entry date range:         {df_verify['entry_date'].min()} to {df_verify['entry_date'].max()}")
print(f"  Mean annualized return:   {df_verify['annualized_return_pct'].mean():.2f}%")
print(f"  Mean days held:           {df_verify['days_held'].mean():.1f}")
print(f"  Mean stop-loss %:         {df_verify['stop_loss_pct'].mean():.2f}%")

# Label shift verification
print(f"\nLabel Shift Verification:")
nan_labels = df_verify['conviction_label'].isna().sum()
n_tickers = df_verify['Ticker'].nunique()
print(f"  NaN in conviction_label: {nan_labels} (expect ~{n_tickers} = 1 per ticker)")

if nan_labels == n_tickers:
    print("  ✓ Label shift correct: exactly 1 NaN per ticker")
else:
    print(f"  ⚠ Expected {n_tickers} NaN, got {nan_labels} — inspect closer")

print("\n" + "=" * 80)
print("✅ VERIFICATION COMPLETE — File ready for Code 7")
print("=" * 80)
print("\n📝 Reminder for Code 7:")
print("   Merge on: features.Date == data_exit_all.entry_date")
print("   Target:   conviction_label  (shifted = no look-ahead bias)")
print("   Drop rows where conviction_label is NaN before training")

VERIFYING data_exit_all.csv

Shape: (2417660, 21)

Columns (21):
  • Ticker: object
  • entry_date: datetime64[ns]
  • entry_price_raw: float64
  • entry_price_adjusted: float64
  • atr_at_entry: float64
  • atr_source: object
  • stop_loss_threshold_rupees: float64
  • stop_loss_pct: float64
  • exit_date: datetime64[ns]
  • exit_price_raw: float64
  • exit_price_adjusted: float64
  • exit_reason: object
  • max_price_achieved: float64
  • drawdown_at_exit_rupees: float64
  • drawdown_at_exit_pct: float64
  • days_held: int64
  • transaction_cost_pct: float64
  • raw_return_pct: float64
  • annualized_return_pct: float64
  • data_quality_flag: object
  • conviction_label: object

First 5 rows:
      Ticker entry_date  entry_price_raw  entry_price_adjusted  atr_at_entry  \
0  3IINFOLTD 2007-01-02        859.41040             861.55896           0.0   
1  3IINFOLTD 2007-01-03        976.27216             978.71290           0.0   
2  3IINFOLTD 2007-01-04        993.09650             995

---
## Cell 14: Helper Function — Inspect Any Stock

In [14]:
# Cell 14: Helper to view individual stock results

def view_stock_results(ticker_symbol, results_df=None, filename='data_exit_all.csv'):
    """
    Display detailed results for any ticker.
    """
    if results_df is None:
        try:
            results_df = pd.read_csv(filename, parse_dates=['entry_date', 'exit_date'])
        except FileNotFoundError:
            print(f"⚠️ File '{filename}' not found")
            return None

    stock_data = results_df[results_df['Ticker'] == ticker_symbol].copy()

    if len(stock_data) == 0:
        print(f"⚠️ No data for: {ticker_symbol}")
        print(f"\nAvailable tickers (first 20):")
        print(sorted(results_df['Ticker'].unique())[:20])
        return None

    print("=" * 80)
    print(f"RESULTS FOR: {ticker_symbol}")
    print("=" * 80)
    print(f"Total trades: {len(stock_data):,}")
    print(f"Date range: {stock_data['entry_date'].min()} to {stock_data['entry_date'].max()}")

    print("\nConviction Distribution (SHIFTED — model target):")
    n = len(stock_data)
    for label in ['High', 'Medium', 'Low', 'Ignore']:
        count = (stock_data['conviction_label'] == label).sum()
        pct = count / n * 100
        print(f"  {label:8s}: {count:>5d} ({pct:>5.2f}%)")
    nan_count = stock_data['conviction_label'].isna().sum()
    print(f"  NaN     : {nan_count:>5d} ({nan_count/n*100:>5.2f}%)")

    print("\nExit Reasons:")
    for r, c in stock_data['exit_reason'].value_counts().items():
        print(f"  {r:25s}: {c:>5d}")

    print("\nReturn Stats:")
    print(stock_data[['annualized_return_pct', 'days_held', 'stop_loss_pct']].describe())

    print("\nSample High Conviction Trades:")
    high = stock_data[stock_data['conviction_label'] == 'High'].head(5)
    if len(high) > 0:
        cols = ['entry_date', 'exit_date', 'days_held',
                'stop_loss_pct', 'annualized_return_pct',
                'conviction_label']
        print(high[cols].to_string(index=False))
    else:
        print("  (None)")

    return stock_data

print("✓ Helper function ready")
print("\nUsage:")
print("  view_stock_results('BPCL')")
print("  view_stock_results('RELIANCE')")

✓ Helper function ready

Usage:
  view_stock_results('BPCL')
  view_stock_results('RELIANCE')


---
## ✅ Summary

### Pipeline:
1. Load `data_features.parquet` (8 cols only) from Drive
2. Fill partial ATR for first ~13 days per stock
3. Simulate buy-hold-exit per Code 8c logic
4. Assign conviction labels by annualized return
5. **Shift labels up by 1 per ticker** (prevents look-ahead bias)
6. Save + auto-download `data_exit_all.csv`

### Output Columns (20 total):
- `Ticker`, `entry_date`
- `entry_price_raw`, `entry_price_adjusted`
- `atr_at_entry`, `atr_source`
- `stop_loss_threshold_rupees`, `stop_loss_pct`
- `exit_date`, `exit_price_raw`, `exit_price_adjusted`, `exit_reason`
- `max_price_achieved`, `drawdown_at_exit_rupees`, `drawdown_at_exit_pct`
- `days_held`, `transaction_cost_pct`
- `raw_return_pct`, `annualized_return_pct`
- `data_quality_flag`
- **`conviction_label`** — SHIFTED label = label for trade at NEXT entry_date (ML target)

### For Code 7:
- Merge: `features.Date == data_exit_all.entry_date`
- ML target: `conviction_label` (already aligned for no look-ahead)
- Drop rows where `conviction_label` is NaN before training

**Ready to run! 🚀**